In [1]:
import json
import numpy as np
import os
import shutil

In [2]:
with open('rcsb_processed_targets/manifest.json', 'r') as f:
    manifest = json.load(f)

In [3]:
os.makedirs('mhc_sample/structures/', exist_ok=True)

In [ ]:
anomalies_dropped = 0

for sample in manifest:
    pdb_id = sample.get('id')
    valid_chain_ids = []
    
    # A lógica do Ernest define que cadeias válidas terminam com '1' neste contexto
    peptide_chain_name = sample.get('peptide_chain', '') + '1'
    protein_chain_name = sample.get('protein_chains', [''])[0] + '1'
    
    # 1. Identificar os IDs das cadeias que importam
    for chain in sample['chains']:
        if chain['chain_name'] in [peptide_chain_name, protein_chain_name]:
            valid_chain_ids.append(chain['chain_id'])
            
    # Validação de sanidade: Precisamos de exatamente 2 cadeias
    if len(valid_chain_ids) != 2:
        print(f"[{pdb_id}] Dropado: Número incorreto de cadeias alvo.")
        anomalies_dropped += 1
        continue
        
    # 2. Carregar o arquivo .npz bruto
    raw_npz_path = os.path.join(raw_npz_dir, f"{pdb_id}.npz")
    if not os.path.exists(raw_npz_path):
        continue
        
    npz_data = dict(np.load(raw_npz_path))
    
    # 3. Atualizar a máscara booleana 
    # True apenas para os índices correspondentes ao MHC e Peptídeo
    for index in range(len(npz_data['mask'])):
        if index in valid_chain_ids:
            npz_data['mask'][index] = True
        else:
            npz_data['mask'][index] = False
            
    # 4. Salvar o .npz limpo
    clean_npz_path = os.path.join(clean_npz_dir, f"{pdb_id}.npz")
    np.savez(clean_npz_path, **npz_data)
    
print(f"Processamento concluído. {anomalies_dropped} estruturas ruidosas dropadas.")

1082

In [118]:
# !zip -r mhc.zip mhc_targets/

In [ ]:
# scp mhc.zip eglukhov@nabu5.ams.stonybrook.edu:/home/eglukhov/projects/boltz/train_data/

In [ ]:
< 2021-09-30 - train
< 2023-01-13 - val
> - test